In [8]:
from pyspark.sql import functions as F

# 1. Silver transform
BRONZE = "checkin_bronze"
SILVER = "checkin_silver"

df_checkin = spark.table(BRONZE)

StatementMeta(, 4d105609-21c8-42e6-aa2a-7ee09ff9236c, 10, Finished, Available, Finished, False)

In [9]:
base_cols = [
    "business_id",
    "date",
    "_ingest_ts",
    "_source_file",
    "_batch_id"
]

df_checkin_silver = (
    df_checkin
    .select(*base_cols)
    .filter(F.col("business_id").isNotNull())
    .filter(F.col("date").isNotNull())
    .dropDuplicates(["business_id"])
)

StatementMeta(, 4d105609-21c8-42e6-aa2a-7ee09ff9236c, 11, Finished, Available, Finished, False)

In [10]:
(
    df_checkin_silver.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(SILVER)

)

StatementMeta(, 4d105609-21c8-42e6-aa2a-7ee09ff9236c, 12, Finished, Available, Finished, False)

In [11]:
print("checkin_silver rows = ", spark.table("checkin_silver").count())

StatementMeta(, 4d105609-21c8-42e6-aa2a-7ee09ff9236c, 13, Finished, Available, Finished, False)

checkin_silver rows =  131930


In [12]:
mssparkutils.notebook.exit("SUCCESS")

StatementMeta(, 4d105609-21c8-42e6-aa2a-7ee09ff9236c, 14, Finished, Available, Finished, False)

ExitValue: SUCCESS

In [13]:
#2. Data Quality Check

df_checkin_silver_DQ = spark.table("checkin_silver")

dup = (
    df_checkin_silver_DQ
    .groupBy("business_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("duplicate business_id =", dup)

df_checkin_silver_DQ.select(
    F.sum(F.col("business_id").isNull().cast("int")).alias("null_business_id"),
    F.sum(F.col("date").isNull().cast("int")).alias("null_date")
).show()

StatementMeta(, 4d105609-21c8-42e6-aa2a-7ee09ff9236c, 15, Finished, Available, Finished, True)

duplicate business_id = 0
+----------------+---------+
|null_business_id|null_date|
+----------------+---------+
|               0|        0|
+----------------+---------+

